In [1]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"준비 완료! 현재 사용 장치: {device}")

준비 완료! 현재 사용 장치: mps


1. 라이브러리 불러오기

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

2. 전처리 선언

In [3]:
# CIFAR-10 전처리 (학습용: 증강 포함)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),           # 좌우 반전 (데이터 증강)
    transforms.RandomCrop(32, padding=4),        # 랜덤 크롭
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),   # RGB 평균
                         (0.2675, 0.2565, 0.2761))   # RGB 표준편차
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761))
])

3. 데이터셋 불러오기

In [4]:

train_dataset = datasets.CIFAR100('./data', train=True,  download=True, transform=train_transform)
test_dataset  = datasets.CIFAR100('./data', train=False, download=True, transform=test_transform)

4. 데이터로더 준비

In [5]:
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, 
                          num_workers=4, pin_memory=True, persistent_workers=True)

5. 클래스 분류 정의

In [6]:
print(f"학습 데이터: {len(train_dataset)}장")
print(f"테스트 데이터: {len(test_dataset)}장")

학습 데이터: 50000장
테스트 데이터: 10000장


6. 모델 설계

In [7]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 합성곱 블록 1: 3채널 → 32채널
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 32×32 → 16×16
        )
        # 합성곱 블록 2: 32채널 → 64채널
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 16×16 → 8×8
        )
        # 합성곱 블록 3: 64채널 → 128채널
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 8×8 → 4×4
        )
        # 완전연결층
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 100)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = CIFAR_CNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {total_params:,}")

파라미터 수: 1,194,084


7. 학습

In [8]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset) * 100
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset) * 100
    return avg_loss, accuracy

EPOCHS = 100
best_acc = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    scheduler.step()
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | "
        f"Test Loss: {test_loss:.4f} Acc: {test_acc:.1f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_cifar_model.pth')

print(f"\n최고 테스트 정확도: {best_acc:.2f}%")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch  1/100 | Train Loss: 3.9780 Acc: 8.9% | Test Loss: 3.4329 Acc: 17.6%
Epoch  2/100 | Train Loss: 3.4272 Acc: 16.8% | Test Loss: 3.0159 Acc: 25.5%
Epoch  3/100 | Train Loss: 3.1645 Acc: 21.2% | Test Loss: 2.7892 Acc: 28.8%
Epoch  4/100 | Train Loss: 3.0055 Acc: 24.7% | Test Loss: 2.6196 Acc: 32.7%
Epoch  5/100 | Train Loss: 2.8842 Acc: 26.7% | Test Loss: 2.5524 Acc: 33.9%
Epoch  6/100 | Train Loss: 2.7946 Acc: 28.3% | Test Loss: 2.4381 Acc: 36.0%
Epoch  7/100 | Train Loss: 2.7241 Acc: 29.6% | Test Loss: 2.3169 Acc: 39.1%
Epoch  8/100 | Train Loss: 2.6495 Acc: 31.3% | Test Loss: 2.2883 Acc: 39.3%
Epoch  9/100 | Train Loss: 2.5913 Acc: 32.8% | Test Loss: 2.2420 Acc: 40.8%
Epoch 10/100 | Train Loss: 2.5572 Acc: 33.5% | Test Loss: 2.2498 Acc: 40.2%
Epoch 11/100 | Train Loss: 2.5124 Acc: 34.4% | Test Loss: 2.1751 Acc: 42.0%
Epoch 12/100 | Train Loss: 2.4712 Acc: 35.5% | Test Loss: 2.1368 Acc: 43.0%
Epoch 13/100 | Train Loss: 2.4364 Acc: 35.9% | Test Loss: 2.0664 Acc: 44.3%
Epoch 14/100 